# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaikhabdullahwaseem17-byte/vigilant-meme/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [2]:
# Clone the repo into /content and switch to it so imports work
import os, sys
from pathlib import Path

REPO = "https://github.com/shaikhabdullahwaseem17-byte/vigilant-meme.git"
ROOT = Path("/content/vigilant-meme")

if not ROOT.exists():
    !git clone {REPO} {ROOT}
else:
    print("Repo already present at", ROOT)

# Change working dir to repo root and add it to Python path
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("CWD:", Path.cwd())
print("Added to sys.path:", sys.path[0])
# now you can import repo modules
import scripts.ml_utils as ml_utils
print("ml_utils loaded, PROCESSED_DIR:", ml_utils.PROCESSED_DIR)


Cloning into '/content/vigilant-meme'...
remote: Enumerating objects: 157, done.
remote: Counting objects: 100% (157/157), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 157 (delta 62), reused 98 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (157/157), 1.86 MiB | 17.81 MiB/s, done.
Resolving deltas: 100% (62/62), done.
CWD: /content/vigilant-meme
Added to sys.path: /content/vigilant-meme
ml_utils loaded, PROCESSED_DIR: /content/vigilant-meme/data/processed


In [3]:
# 1) Setup: imports, seed, method choice (run this cell first)
import os
import sys
from pathlib import Path
import random
import numpy as np
import pandas as pd

# scikit-learn imports used later
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

# repo utilities
import scripts.ml_utils as ml_utils

# reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Environment ready. Seed =", SEED)
print("Primary method: LogisticRegression (interpretable). Backup / non-linear check: RandomForest.")

Environment ready. Seed = 42
Primary method: LogisticRegression (interpretable). Backup / non-linear check: RandomForest.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [6]:
# 2) Split design: prepare/load features and create stratified train/val/test split
from pathlib import Path
prepared_path = Path("data/processed/refresh_feature_vector.csv")
raw_path = Path("data/raw/content_refresh_anonymized.csv")

# If prepared features do not exist, run the repo prepare script (writes CSV)
if not prepared_path.exists():
    if not raw_path.exists():
        raise FileNotFoundError(f"Raw data not found at {raw_path}. Put the CSV in data/raw/ or run prepare script manually.")
    print("Prepared features not found — running scripts/01_prepare_features.py to create them.")
    !python scripts/01_prepare_features.py

# Load prepared features
df = pd.read_csv(prepared_path)
print("Prepared features rows:", len(df))

# Target the repo uses
if 'is_declining_label' not in df.columns:
    df['is_declining_label'] = df.get('trend_direction','').astype(str).str.lower().eq('down').astype(int)

# Choose numeric features from ml_utils list available in the DF
feature_cols = [c for c in ml_utils.MODEL_NUMERIC_FEATURES if c in df.columns]
if not feature_cols:
    # fallback minimal features
    fallback = []
    if 'log_impressions_90d' in df.columns:
        fallback.append('log_impressions_90d')
    if 'engagement_rate' in df.columns:
        fallback.append('engagement_rate')
    if not fallback:
        # last resort
        fallback = [c for c in ['impressions_90d','engagement_rate'] if c in df.columns]
    feature_cols = fallback

print("Using features (first 10):", feature_cols[:10])

# stratified split (train/val/test) preserving base rate
if df['is_declining_label'].nunique() < 2:
    raise ValueError("Target has only one class in prepared data. Can't stratify. Check data or labels.")

train_val, test = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df['is_declining_label'])
train, val = train_test_split(train_val, test_size=0.25, random_state=SEED, stratify=train_val['is_declining_label'])

print("Split sizes (train, val, test):", len(train), len(val), len(test))

# prepare X/y for model training
X_train = train[feature_cols].fillna(0).values
y_train = train['is_declining_label'].values
X_val = val[feature_cols].fillna(0).values
y_val = val['is_declining_label'].values
X_test = test[feature_cols].fillna(0).values
y_test = test['is_declining_label'].values

Prepared features not found — running scripts/01_prepare_features.py to create them.
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/vigilant-meme/data/processed/refresh_feature_vector.csv
Prepared features rows: 30000
Using features (first 10): ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions']
Split sizes (train, val, test): 18000 6000 6000


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [7]:
# 3) Train + compare vs baseline: compute baseline score, train models, compute precision@K and AUC
from scripts.ml_utils import percentile_rank, normalize, precision_at_k

# Compute baseline components (same as scripts/02_baseline_score.py)
train_df = df  # we computed baseline on full df earlier; but we will evaluate on test below
df['visibility_score'] = percentile_rank(np.log1p(df['impressions_90d']))
df['freshness_risk_score'] = percentile_rank(df['days_since_last_update'] if 'days_since_last_update' in df.columns else df['content_age_days'])
df['position_opportunity_score'] = ((1 - normalize(df['avg_position'].clip(lower=1, upper=50))) * df['visibility_score'] * (df.get('avg_position', 0) > 0).astype(int))
df['depth_gap_score'] = (1 - percentile_rank(df['word_count'])) * df['visibility_score']
df['baseline_refresh_score'] = (0.40 * df['visibility_score'] + 0.30 * df['freshness_risk_score'] + 0.25 * df['position_opportunity_score'] + 0.05 * df['depth_gap_score']).clip(0,1)

# Recompute baseline on the test split specifically so ranking is fair
test = test.copy().reset_index(drop=True)
test['visibility_score'] = percentile_rank(np.log1p(test['impressions_90d']))
test['freshness_risk_score'] = percentile_rank(test['days_since_last_update'] if 'days_since_last_update' in test.columns else test['content_age_days'])
test['position_opportunity_score'] = ((1 - normalize(test['avg_position'].clip(lower=1, upper=50))) * test['visibility_score'] * (test.get('avg_position', 0) > 0).astype(int))
test['depth_gap_score'] = (1 - percentile_rank(test['word_count'])) * test['visibility_score']
test['baseline_refresh_score'] = (0.40 * test['visibility_score'] + 0.30 * test['freshness_risk_score'] + 0.25 * test['position_opportunity_score'] + 0.05 * test['depth_gap_score']).clip(0,1)

# Train Logistic Regression
clf_lr = LogisticRegression(random_state=SEED, max_iter=2000)
clf_lr.fit(X_train, y_train)
probs_lr = clf_lr.predict_proba(X_test)[:,1]
auc_lr = roc_auc_score(y_test, probs_lr) if len(np.unique(y_test))>1 else float('nan')

# Train Random Forest
clf_rf = RandomForestClassifier(n_estimators=200, random_state=SEED)
clf_rf.fit(X_train, y_train)
probs_rf = clf_rf.predict_proba(X_test)[:,1]
auc_rf = roc_auc_score(y_test, probs_rf) if len(np.unique(y_test))>1 else float('nan')

# Evaluate precision@K for baseline and models on the test split
ks = [20, 50]
rows = []
for k in ks:
    baseline_prec = precision_at_k(test['is_declining_label'].values, test['baseline_refresh_score'].values, k)
    lr_prec = precision_at_k(test['is_declining_label'].values, probs_lr, k)
    rf_prec = precision_at_k(test['is_declining_label'].values, probs_rf, k)
    rows.append({'k': k, 'baseline_prec': baseline_prec, 'lr_prec': lr_prec, 'rf_prec': rf_prec, 'auc_lr': auc_lr, 'auc_rf': auc_rf, 'base_rate': test['is_declining_label'].mean()})

results_df = pd.DataFrame(rows)
results_df

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,k,baseline_prec,lr_prec,rf_prec,auc_lr,auc_rf,base_rate
0,20,0.40,0.95,0.95,0.690094,0.768331,0.542
1,50,0.48,0.88,0.98,0.690094,0.768331,0.542


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [8]:
# 4) Errors and interpretation: coefficients, permutation importance, and example mistakes
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance

# Logistic regression coefficients
coef_df = pd.DataFrame({'feature': feature_cols, 'coef': clf_lr.coef_[0]})
coef_df['abs_coef'] = coef_df['coef'].abs()
coef_df = coef_df.sort_values('abs_coef', ascending=False).drop(columns='abs_coef')
print("Top logistic regression coefficients:")
display(coef_df.head(10))

# Permutation importance for RandomForest on validation set
perm = permutation_importance(clf_rf, X_val, y_val, n_repeats=30, random_state=SEED, n_jobs=-1)
perm_df = pd.DataFrame({'feature': feature_cols, 'imp_mean': perm.importances_mean, 'imp_std': perm.importances_std}).sort_values('imp_mean', ascending=False)
print("Top permutation importances (RandomForest):")
display(perm_df.head(10))

# Error analysis: false positives and false negatives (RandomForest predictions on test)
test_with_preds = test.copy().reset_index(drop=True)
test_with_preds['prob_rf'] = probs_rf
test_with_preds['prob_lr'] = probs_lr
test_with_preds['baseline_score'] = test_with_preds['baseline_refresh_score']

# False positives: high prob but label == 0
false_positives = test_with_preds[(test_with_preds['prob_rf'] >= 0.5) & (test_with_preds['is_declining_label'] == 0)].sort_values('prob_rf', ascending=False).head(5)
# False negatives: label == 1 but low prob
false_negatives = test_with_preds[(test_with_preds['prob_rf'] < 0.5) & (test_with_preds['is_declining_label'] == 1)].sort_values('prob_rf').head(5)

print("Top false positives (review these):")
display(false_positives[['content_id','prob_rf','prob_lr','baseline_score','impressions_90d','engagement_rate','is_declining_label']])

print("Top false negatives (review these):")
display(false_negatives[['content_id','prob_rf','prob_lr','baseline_score','impressions_90d','engagement_rate','is_declining_label']])

# Short guidance notes to paste in the markdown above the cell:
print("\nInterpretation notes (paste into the markdown above):")
print("- Top features: " + ", ".join(list(coef_df.head(3)['feature'])))
print("- The model leans on impressions (visibility) and engagement-related features. False positives often are high-impression pages that are not labeled declining; false negatives are low-impression but quickly-declining pages the model missed.")

Top logistic regression coefficients:


,feature,coef
5,log_impressions_90d,0.253479
6,log_clicks_90d,-0.166486
7,log_sessions_90d,-0.146179
13,ctr,-0.029360
1,competition,0.014064
10,days_with_sessions,-0.013617
14,avg_position,-0.009915
15,engagement_rate,-0.007763
17,ai_traffic_pct,-0.006785
2,cpc,-0.004122


Top permutation importances (RandomForest):


,feature,imp_mean,imp_std
9,days_with_impressions,0.064317,0.003976
14,avg_position,0.042283,0.003400
11,content_age_days,0.035883,0.003957
5,log_impressions_90d,0.023867,0.003292
16,scroll_rate,0.009972,0.002126
13,ctr,0.009444,0.002868
6,log_clicks_90d,0.007850,0.002450
4,char_count,0.006417,0.002011
12,days_since_last_update,0.006278,0.002253
7,log_sessions_90d,0.004011,0.002387


Top false positives (review these):


,content_id,prob_rf,prob_lr,baseline_score,impressions_90d,engagement_rate,is_declining_label
892,content_25a763874cf0,0.965,0.714250,0.544765,908,0.00,0
5777,content_959e1cc9feeb,0.940,0.725563,0.484254,611,0.00,0
1427,content_54e9fbc9f066,0.940,0.600051,0.711832,32722,2.56,0
5486,content_7dcd5d29a6f2,0.935,0.695568,0.430663,187,0.00,0
563,content_9824710082d8,0.935,0.729980,0.467086,283,0.00,0


Top false negatives (review these):


,content_id,prob_rf,prob_lr,baseline_score,impressions_90d,engagement_rate,is_declining_label
5999,content_b0a5c92e100b,0.010,0.155386,0.110357,1,0.0,1
4340,content_c0af3d6f9dd3,0.010,0.085551,0.128844,2,0.0,1
3241,content_d82b8750bca8,0.055,0.393234,0.470648,1137,0.0,1
3350,content_fa60ce0995f1,0.060,0.429708,0.481449,2481,0.0,1
1684,content_a55d958ec725,0.065,0.330124,0.145584,3,0.0,1



Interpretation notes (paste into the markdown above):
- Top features: log_impressions_90d, log_clicks_90d, log_sessions_90d
- The model leans on impressions (visibility) and engagement-related features. False positives often are high-impression pages that are not labeled declining; false negatives are low-impression but quickly-declining pages the model missed.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.